## Data Cleaning and Preprocessing
The data used is a food delivery orders dataset. Issues in the dataset include inconsistent data types, missing values, irregular categorical entries, textual noise, duplications, and potential outliers. The objective here is to impose structure, correctness, and analytical readiness on the raw data so that it may be reliably used for subsequent tasks.

### Preliminary Examination of the Data
A thorough initial exploration is essential for understanding the structure and condition of the dataset before applying cleaning operations.
This stage seeks to identify data quality issues early and develop an informed strategy for resolving them.

#### Importing Libraries and Loading the Data
Essential Python libraries for data manipulation, statistical summarisation, and visual inspection include libraries such as Pandas, NumPy, Matplotlib, and Seaborn. These libraries form the foundational tools needed to conduct a methodical and reproducible cleaning workflow.

In [0]:
import pandas as pd  # Core library for data manipulation and preprocessing
import numpy as np  # Numerical computing library used for arrays and mathematical operations
import matplotlib.pyplot as plt  # Plotting library for generating static visualisations
import seaborn as sns  # Statistical visualisation library built on top of matplotlib

In [0]:
df = pd.read_csv('/Volumes/epgp_aai_aai/ai_schema/ai_volume/eda-data-sets/food_delivery_dataset.csv')  # Load the raw food delivery dataset into a pandas DataFrame

In [0]:
df.head()  # Top few rows of the data frame

In [0]:
# Subset of features used
features = ['order_id', 'price', 'delivery_rating', 'delivery_time_minutes',
            'customer_city', 'order_date', 'restaurant_name', 'food_item', 'quantity',
            'discount', 'order_status', 'payment_method']

df = df[features]

In [0]:
df.head()  # Top few rows of the data frame

#### Inspecting Data Types and Structural Properties
A concise summary of feature data types, extent of missing values, memory usage, and structural metadata helps in better understanding the data. This inspection helps identify variables stored in inappropriate formats and highlights fields that require type correction or imputation.

In [0]:
df.info()  # View data types and summary of null values

In [0]:
df.shape  # Examining the shape and size of the data

#### Examining Summary Statistics
A descriptive statistical overview of both numerical and categorical variables helps in further understanding the data. This step provides insight into the distribution of numerical features, central tendencies and dispersion, frequency patterns within categorical fields, and potential anomalies, outliers, or irregularities. Such summaries allow us to assess the dataset’s internal consistency and detect values that warrant deeper investigation.

In [0]:
df.describe(include = 'all')  # View descriptive statistics for all columns

#### Quantifying Missing Values
A detailed breakdown of missingness should be computed. This enables us to determine the variables most affected by missing data, patterns of missingness across the dataset, and potential implications for downstream analyses. Understanding the nature and extent of missing data is crucial for choosing suitable imputation or removal strategies.

In [0]:
display(df.isnull().sum()) # Count missing values in each column
missing_values_proposition = df.isnull().sum()/len(df)
#display(missing_values_proposition)

#### Viewing Representative Samples of the Data
Inspecting random subsets of observations is also a good way to get a sense of the data. This provides a more holistic preliminary understanding of the data.

In [0]:
df.sample(10)  # Display a random sample of rows

### Data Cleaning
A systematic data cleaning process ensures that the dataset becomes internally consistent, analytically reliable, and ready for modelling. Steps such as cleaning column names, removing duplicates, correcting corrupted numeric fields, fixing data types, standardising categorical values, handling missing values, and treating outliers, are all part of data cleaning.

#### Cleaning and Standardising Column Names
Standardising column names improves usability, avoids syntax errors, and ensures consistent access patterns across the notebook.

In [0]:
# Standardise column names
df.columns = (df.columns.str.lower().str.strip().str.replace(' ', '_').str.replace('-', '_'))
df.columns  # View updated column names

In [0]:
# Identify numeric and categorical columns
num_cols = df.select_dtypes(include = ['int64', 'float64']).columns
cat_cols = df.select_dtypes(include = ['object']).columns

print('Numerical features =', num_cols)
print()
print('Categorical features = ', cat_cols)

#### Removing Duplicate Rows
Duplicate rows distort frequency calculations, bias descriptive statistics, and inflate counts. Removing exact duplicates ensures cleaner analysis.

In [0]:
df.drop_duplicates(inplace = True)  # Remove duplicate rows

In [0]:
df.shape  # Examining the shape and size of the data

#### Cleaning Corrupted Numeric Fields
Certain numeric fields contain inconsistent formats such as strings (`'three'`), blanks, `'N/A'`, or symbols. We convert all values to string, replace invalid entries, and coerce into proper numeric types. We fix corrupted numeric fields such as price / quantity / discount.

In [0]:
# Clean quantity column
df['quantity'] = df['quantity'].astype(str)  # Convert to string
df['quantity'] = df['quantity'].replace({'three': 3, ' ': np.nan, '': np.nan})
df['quantity'] = pd.to_numeric(df['quantity'], errors = 'coerce')  # Convert to numeric

When converting a column to numeric or datetime, pandas normally raises an error if it encounters invalid values. Using `errors = 'coerce'` forces pandas to convert valid values normally, turn invalid or non-convertible values (such as `'abc'`, `'three'`, `' '`) into `NaN`. This is useful in data cleaning because it standardises bad entries as missing values, making them easy to detect and impute later.

In [0]:
# Clean price column
df['price'] = df['price'].astype(str)  # Convert to string
df['price'] = df['price'].replace({'N/A': np.nan, ' ': np.nan, '': np.nan})
df['price'] = pd.to_numeric(df['price'], errors = 'coerce')  # Convert to numeric

In [0]:
# Clean discount column
df['discount'] = df['discount'].astype(str)  # Convert to string
df['discount'] = df['discount'].replace({'N/A': np.nan, '5%': 5, ' ': np.nan, '': np.nan})
df['discount'] = pd.to_numeric(df['discount'], errors = 'coerce')  # Convert to numeric

In [0]:
df.sample(2)  # Quick look at the data

#### Fixing Incorrect Data Types
Correcting data types is essential for valid computations, filtering, and statistical analysis. For example, date and time information should be in the right data type format, numeric fields should be numeric data type, and categorical fields should be of the category data type.

In [0]:
# Convert numeric columns
for col in num_cols:
    df[col] = pd.to_numeric(df[col], errors = 'coerce')

In [0]:
# Convert categorical columns
for col in cat_cols:
    df[col] = df[col].astype('category')

In [0]:
# Convert order_date to datetime
df['order_date'] = pd.to_datetime(df['order_date'], errors = 'coerce')

In [0]:
df.info()  # Verify updated data types

#### Standardising String and Categorical Values
We normalise case, remove whitespace, and map inconsistent category labels into standardised, clean labels.

In [0]:
# Normalize text columns considering lowercase, strip, removing tabs/newlines
def normalise_text(col):
    return col.astype(str).str.strip().str.lower().str.replace('\t', ' ').str.replace('\n', ' ')

string_cols = ['customer_city', 'restaurant_name', 'payment_method', 'order_status', 'food_item']

for col in string_cols:
    df[f'{col}_norm'] = normalise_text(df[col])

In [0]:
# Mapping dictionaries for standardising categories
city_map = {'mumbai': 'Mumbai', 'delhi': 'New Delhi', 'bangalore': 'Bengaluru', 'bengaluru': 'Bengaluru',
            'pune': 'Pune', 'nagpur': 'Nagpur', 'hyderabad': 'Hyderabad'}
restaurant_map = {'pizza hub': 'Pizza Hub', 'foodie corner': 'Foodie Corner', 'tasty treats': 'Tasty Treats',
                  'biryani house': 'Biryani House', 'the spice route': 'The Spice Route'}
payment_map = {'upi': 'UPI', 'card': 'Card', 'creditcard': 'Card', 'credit card': 'Card', 'cash': 'Cash', 'cod': 'Cash'}
status_map = {'completed': 'Completed', 'cancelled': 'Cancelled', 'canceled': 'Cancelled', 'returned': 'Returned', 'in-progress': 'In-Progress'}
food_map = {'pizza': 'Pizza', 'burger': 'Burger', 'pasta': 'Pasta', 'biryani': 'Biryani',
            'sandwich': 'Sandwich', 'noodles': 'Noodles', 'momos': 'Momos', 'coke': 'Coke'}

# Apply cleaned mappings
df['customer_city_clean'] = df['customer_city_norm'].map(city_map).fillna(df['customer_city_norm'])
df['restaurant_name_clean'] = df['restaurant_name_norm'].map(restaurant_map).fillna(df['restaurant_name_norm'])
df['payment_method_clean'] = df['payment_method_norm'].map(payment_map).fillna(df['payment_method_norm'])
df['order_status_clean'] = df['order_status_norm'].map(status_map).fillna(df['order_status_norm'])
df['food_item_clean'] = df['food_item_norm'].map(food_map).fillna(df['food_item_norm'])

# Check
df[['customer_city', 'customer_city_norm', 'customer_city_clean']].sample(5)
df[['restaurant_name', 'restaurant_name_norm', 'restaurant_name_clean']].sample(5)

In [0]:
df.sample(2)  # Quick look at the data

In [0]:
# Apply cleaned mappings to main columns and drop redundant columns
df['customer_city'] = df['customer_city_norm'].map(city_map).fillna(df['customer_city_norm'])
df['restaurant_name'] = df['restaurant_name_norm'].map(restaurant_map).fillna(df['restaurant_name_norm'])
df['payment_method'] = df['payment_method_norm'].map(payment_map).fillna(df['payment_method_norm'])
df['order_status'] = df['order_status_norm'].map(status_map).fillna(df['order_status_norm'])
df['food_item'] = df['food_item_norm'].map(food_map).fillna(df['food_item_norm'])
df.drop(['customer_city_norm', 'customer_city_clean', 'restaurant_name_norm', 'restaurant_name_clean',
         'payment_method_norm', 'payment_method_clean', 'order_status_norm', 'order_status_clean',
         'food_item_norm', 'food_item_clean'], axis = 1, inplace = True)

In [0]:
df.sample(2)  # Quick look at the data

#### Handling Missing Values
Missing data needs to be dealt with properly to keep the dataset complete and easy to analyse. We first remove rows where all columns are empty. Then we fill missing values based on the nature of the feature and our understanding of the data. For numeric columns, we can replace missing values with the median, which works well even if there are outliers. For categorical columns, we can replace missing values with the mode, the most common category. This ensures the dataset stays clean and ready for further analysis. Missing values can distort analysis and cause errors in calculations.

In [0]:
df.dropna(how = "all", inplace = True)  # Drop rows where all columns are missing

In [0]:
df.shape  # Examining the shape and size of the data

In [0]:
# Impute numeric columns with median
for col in num_cols:
    df.fillna({col: df[col].median()}, inplace = True)

In [0]:
# Impute categorical columns with mode
for col in cat_cols:
    df.fillna({col: df[col].mode()[0]}, inplace = True)

In [0]:
df.isnull().sum()  # Verify missing values resolved

#### Handling Outliers
Outliers are values that are unusually high or unusually low compared to the rest of the data. These values can affect averages, distort patterns, and lead to misleading insights if not handled properly. To detect outliers, we can use the IQR (interquartile range) method, a common and reliable technique. Here, $Q1$ is the 25th percentile (lower quartile), and $Q3$ is the 75th percentile (upper quartile). The interquartile range is defined as $IQR = Q3 − Q1$. Any value below $Q1 − 1.5 × IQR$ or above $Q3 + 1.5 × IQR$ is treated as a potential outlier.

We can then remove or cap these extreme values to reduce the impact of unusually large or small numbers, make statistical summaries more accurate, and improve the quality of analysis and visualisations. This ensures the dataset reflects typical behaviour while still keeping the underlying distribution meaningful.

This is typically applied on numerical features.

In [0]:
# Visualise outliers using boxplots
plt.figure(figsize = (10, 6))

# Remove outliers using IQR
for k, col in enumerate(num_cols[:4], start = 1):
    plt.subplot(2, 2, k)
    sns.boxplot(x = df[col])
    plt.xticks(fontsize = 8)
    plt.yticks(fontsize = 8)

plt.tight_layout()
plt.show()

Boxplots help us understand the spread, central tendency, and presence of outliers in numerical variables. For instance, the `'price'` column shows many outliers. But before deciding to remove or cap outlier values, we need to make sure that this does not negatively affect the downstream proceses in the pipeline.

In [0]:
# Data that remains if we remove outliers from the price column
df[(df['price'] >= df['price'].quantile(0.25)) & (df['price'] <= df['price'].quantile(0.75))]['price']

In [0]:
df[(df['price'] >= df['price'].quantile(0.25)) & (df['price'] <= df['price'].quantile(0.75))]['price'].describe()

It looks like applying the IQR method to the price column might end up making it redundant. We can choose to retain the column or deal with it differently.

### Filtering Data by Business Objective
The cleaned dataset can then be filtered to focus on recent trends, performance, and operational patterns relevant to the business. Selecting only orders from the last six months for focused analysis in this case makes sense.

In [0]:
df_recent = df[df['order_date'] >= pd.to_datetime('2024-06-01')]  # Filter recent orders

### Exporting the Cleaned Dataset
Save the processed dataset in CSV, Excel, or other formats for downstream use.

In [0]:
df.to_csv('/Volumes/epgp_aai_aai/ai_schema/ai_volume/eda-data-sets/clean_food_delivery_dataset_interim.csv', index = False)  # CSV